In [1]:
import getml
import mlflow
import getml.mlflow

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")

if not mlflow.search_experiments(filter_string="name='interstate94'"):
    mlflow.create_experiment("interstate94")
mlflow.set_experiment("interstate94")

getml.mlflow.autolog()

In [3]:
if getml.engine.is_alive() or getml.engine.is_monitor_alive() or getml.engine.is_engine_alive():
    getml.engine.shutdown()
getml.engine.launch()

Launching ./getML --allow-push-notifications=true --allow-remote-ips=false --home-directory=/home/manuel/.getML --in-memory=true --install=false --launch-browser=true --log=false --project-directory=/home/manuel/.getML/projects in /home/manuel/Projects/github/getml-mlflow/.venv/lib/python3.11/site-packages/getml/.getML/getml-community-1.5.0-amd64-linux...
Launched the getML Engine. The log output will be stored in /home/manuel/.getML/logs/getml_20250114183504.log


In [4]:
getml.engine.set_project("interstate94")

Output()

Connected to project 'interstate94'.

In [5]:
traffic = getml.datasets.load_interstate94(roles=False, units=False)

In [6]:
traffic.set_role("ds", getml.data.roles.time_stamp)
traffic.set_role("holiday", getml.data.roles.categorical)
traffic.set_role("traffic_volume", getml.data.roles.target)

In [7]:
split = getml.data.split.time(traffic, "ds", test=getml.data.time.datetime(2018, 3, 15))

In [8]:
time_series = getml.data.TimeSeries(
    population=traffic,
    split=split,
    time_stamps="ds",
    horizon=getml.data.time.hours(1),
    memory=getml.data.time.days(7),
    lagged_targets=True,
)

pipe = getml.pipeline.Pipeline(
    tags=["memory: 7d", "horizon: 1h", "fast_prop"],
    data_model=time_series.data_model,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[
        getml.feature_learning.FastProp(
            loss_function=getml.feature_learning.loss_functions.SquareLoss,
            num_threads=1,
            num_features=20,
        )
    ],
    predictors=[getml.predictors.XGBoostRegressor()],
)
pipe

2025/01/14 18:35:06 INFO mlflow.tracking._tracking_service.client: 🏃 View run Pipeline at: http://localhost:5000/#/experiments/844494434965818253/runs/cd2cd43dce9c430891cffbe9caeacccc.
2025/01/14 18:35:06 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Pipeline(mlflow_run_id='cd2cd43dce9c430891cffbe9caeacccc',
         data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop'])

In [9]:
pipe.fit(time_series.train)

Engine metrics are available in the Enterprise edition.


Checking data model...

Output()

Engine metrics are available in the Enterprise edition.

OK.

Output()

Trained pipeline.

2025/01/14 18:35:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run fit at: http://localhost:5000/#/experiments/844494434965818253/runs/ea689f8365a94b30b7c181e0c1b2b09d.
2025/01/14 18:35:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


Time taken: 0:00:08.326483.



Pipeline(mlflow_run_id='cd2cd43dce9c430891cffbe9caeacccc',
         data_model='population',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['traffic'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['memory: 7d', 'horizon: 1h', 'fast_prop', 'container-ujDKMZ'])

In [10]:
time_series.data_model.population.roles.target

['traffic_volume']

In [11]:
pipe.score(time_series.test)

Output()

2025/01/14 18:35:24 INFO mlflow.tracking._tracking_service.client: 🏃 View run score at: http://localhost:5000/#/experiments/844494434965818253/runs/cf4c879e97c741bab39cd10e6e77d951.
2025/01/14 18:35:24 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/844494434965818253.


,date time,set used,target,mae,rmse,rsquared
0,2025-01-14 18:35:20,train,traffic_volume,200.4302,299.2045,0.9768
1,2025-01-14 18:35:24,test,traffic_volume,179.9515,269.631,0.9816
